# 🏪 Laboratorio integrador: del dato a la decisión con el caso Mayoristas

## 📖 La historia

Un **distribuidor mayorista de alimentos en Portugal** vende a 440 clientes: hoteles, restaurantes y cafés (el canal **Horeca**) y tiendas de autoservicio y abarrotes (el canal **Retail**). Hoy los atiende a todos igual: el mismo vendedor, el mismo catálogo y el mismo camión.

El director comercial sospecha que eso le cuesta dinero: un restaurante que compra pescado fresco todos los días no necesita lo mismo que una tienda que compra cajas de detergente cada mes. Te entrega una tabla con el **gasto anual de cada cliente en seis categorías** y te pide una propuesta.

Este es el **cierre del curso**: vas a usar, en un solo proyecto, todo lo que aprendiste. Transformar y escalar (sesión 0), reducir con PCA (sesión 3), agrupar con K-Means (sesión 1), detectar atípicos con DBSCAN (sesión 2) y, sobre todo, **traducir los clusters a decisiones**.

> ### ❓ ¿Cómo debe organizar el distribuidor a sus clientes mayoristas: qué equipos de venta, qué surtido y qué logística le tocan a cada grupo?

## 🧭 Lo que vas a aprender

1. A reconocer datos de gasto **muy sesgados** y por qué conviene aplicar `log1p` **antes** de escalar.
2. A usar **PCA** como paso intermedio: resumir 6 categorías en 2 ejes que se pueden interpretar y graficar.
3. A combinar **K-Means** (segmentos) y **DBSCAN** (clientes atípicos) sobre el mismo espacio.
4. A **validar** un agrupamiento con una etiqueta que **no** usaste para agrupar (el canal de venta).
5. A convertir cada segmento en una **recomendación de negocio** con números en las unidades originales.

## ⏱️ Agenda (3 horas)

| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación, recap del curso y carga de datos | 10 min |
| 1 | Exploración: distribuciones sesgadas y outliers enormes | 20 min |
| 2 | ¿Por qué `log1p` antes de escalar? | 20 min |
| 3 | PCA: de 6 categorías a 2 ejes con significado | 25 min |
| ☕ | Descanso | 10 min |
| 4 | K-Means sobre los componentes principales | 25 min |
| 5 | DBSCAN: ¿quién no encaja en ningún segmento? | 20 min |
| 6 | Perfil de los segmentos en unidades originales | 20 min |
| 7 | Validación contra el canal de venta | 15 min |
| 8 | Recomendaciones de negocio y cierre | 15 min |
| | **Total** | **180 min** |

## 📋 Las variables

Datos: *Wholesale customers* (UCI Machine Learning Repository, Cardoso, 2014; licencia CC BY 4.0). Los gastos están en **unidades monetarias anuales** (u.m.).

| Variable (original) | En el notebook | Descripción | Unidades |
|---|---|---|---|
| `Channel` | `canal` | Horeca (hoteles, restaurantes, cafés) o Retail | categoría |
| `Region` | `region` | Lisboa, Oporto u Otra | categoría |
| `Fresh` | `frescos` | Gasto anual en productos frescos (frutas, verduras, carne, pescado) | u.m./año |
| `Milk` | `lacteos` | Gasto anual en lácteos | u.m./año |
| `Grocery` | `abarrotes` | Gasto anual en abarrotes (productos secos de anaquel) | u.m./año |
| `Frozen` | `congelados` | Gasto anual en congelados | u.m./año |
| `Detergents_Paper` | `detergentes_papel` | Gasto anual en detergentes y productos de papel | u.m./año |
| `Delicassen` | `delicatessen` | Gasto anual en delicatessen (embutidos, quesos finos) | u.m./año |

> ⚠️ **Regla del caso:** `canal` y `region` **no** entran al agrupamiento. Sólo agrupamos con los seis gastos. El canal lo guardamos para el final, como "examen" de nuestros segmentos.

---
# 0️⃣ Preparación (10 min)

## 📋 Recap: lo que ya sabes de las sesiones 1 a 3

| Sesión | Técnica | La idea clave | La trampa que aprendiste a evitar |
|---|---|---|---|
| 1 | **K-Means** | Asigna cada punto al centroide más cercano y recalcula; tú eliges `k` | Elegir `k` sólo con el codo; olvidar `n_init` y `random_state` |
| 2 | **DBSCAN** | Un cluster es una zona densa; lo que no cabe en ninguna es **ruido** (-1) | Elegir `eps` al azar en vez de usar la gráfica de k-distancia |
| 3 | **PCA** | Rota los datos para que pocos ejes (componentes) concentren la varianza | Aplicarlo sin escalar; no leer las cargas |
| Todas | **Escalado** | Las distancias y la varianza dependen de las unidades | Agrupar con variables en escalas distintas |

Hoy los encadenamos: **preparar → reducir → agrupar → validar → interpretar → actuar**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_silueta_k(tabla)` | Silhouette promedio según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import HTML, display

pio.renderers.default = 'plotly_mimetype'

def mostrar_figura(fig):
    display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        mostrar_figura(fig)


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    mostrar_figura(fig)


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    mostrar_figura(fig)


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    mostrar_figura(fig)


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    mostrar_figura(fig)


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    mostrar_figura(fig)


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    mostrar_figura(fig)


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    mostrar_figura(fig)


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    mostrar_figura(fig)


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    mostrar_figura(fig)


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    mostrar_figura(fig)


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    mostrar_figura(fig)

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido y silhouette (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y silhouette en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [2]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Silhouette de un agrupamiento. El ruido (-1) se excluye del cálculo,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
    else:
        fila['silhouette'] = np.nan
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Cargar los datos

El archivo viene en la carpeta `datos/` junto a este notebook. Renombramos las columnas al español y traducimos los códigos de `canal` y `region` para leerlos más fácil.

In [3]:
# Esta celda va de regalo: léela con calma y ejecútala
import pandas as pd
import numpy as np

df = pd.read_csv('datos/wholesale_customers.csv')

df = df.rename(columns={
    'Channel': 'canal',
    'Region': 'region',
    'Fresh': 'frescos',
    'Milk': 'lacteos',
    'Grocery': 'abarrotes',
    'Frozen': 'congelados',
    'Detergents_Paper': 'detergentes_papel',
    'Delicassen': 'delicatessen',
})
df['canal'] = df['canal'].map({1: 'Horeca', 2: 'Retail'})
df['region'] = df['region'].map({1: 'Lisboa', 2: 'Oporto', 3: 'Otra'})

# Las seis variables de gasto: son las ÚNICAS que entran al agrupamiento
gasto = ['frescos', 'lacteos', 'abarrotes', 'congelados', 'detergentes_papel', 'delicatessen']

df.head()

,canal,region,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen
0,Retail,Otra,12669,9656,7561,214,2674,1338
1,Retail,Otra,7057,9810,9568,1762,3293,1776
2,Retail,Otra,6353,8808,7684,2405,3516,7844
3,Horeca,Otra,13265,1196,4221,6404,507,1788
4,Retail,Otra,22615,5410,7198,3915,1777,5185


---
# 1️⃣ Exploración: distribuciones sesgadas y outliers enormes (20 min)

Antes de agrupar hay que conocer los datos: cuántos son, si hay nulos, en qué rango se mueve cada gasto y cómo se relacionan entre sí.

In [4]:
print('Forma:', df.shape)
print('Valores nulos:')
print(df.isnull().sum())


Forma: (440, 8)
Valores nulos:
canal                0
region               0
frescos              0
lacteos              0
abarrotes            0
congelados           0
detergentes_papel    0
delicatessen         0
dtype: int64


In [5]:
print(df[gasto].describe().round(0))


        frescos  lacteos  abarrotes  congelados  detergentes_papel  \
count     440.0    440.0      440.0       440.0              440.0   
mean    12000.0   5796.0     7951.0      3072.0             2881.0   
std     12647.0   7380.0     9503.0      4855.0             4768.0   
min         3.0     55.0        3.0        25.0                3.0   
25%      3128.0   1533.0     2153.0       742.0              257.0   
50%      8504.0   3627.0     4756.0      1526.0              816.0   
75%     16934.0   7190.0    10656.0      3554.0             3922.0   
max    112151.0  73498.0    92780.0     60869.0            40827.0   

       delicatessen  
count         440.0  
mean         1525.0  
std          2820.0  
min             3.0  
25%           408.0  
50%           966.0  
75%          1820.0  
max         47943.0  


In [6]:
pd.crosstab(df['region'], df['canal'], margins=True)


canal,Horeca,Retail,All
region,,,
Lisboa,59,18,77
Oporto,28,19,47
Otra,211,105,316
All,298,142,440


✍️ **¿Qué observas?** ¿Hay nulos? Compara la media y la mediana (50%) de cada gasto: ¿qué te dice que la media sea mucho mayor? ¿Cuánto vale el máximo comparado con la mediana? ¿Qué canal domina?

No hay nulos. Las medias son mucho mayores que las medianas, por ejemplo frescos 12000 frente a 8504; delicatessen tiene máximo 47943. Hay pocos clientes muy grandes y el canal Horeca domina con 298 de 440 clientes.

In [7]:
plot_distributions(df, gasto)


In [8]:
# Esta celda va de regalo: léela con calma y ejecútala
# Dos números que resumen el sesgo de cada variable
sesgo = pd.DataFrame({
    'asimetria': df[gasto].skew().round(2),          # 0 = simétrica; > 1 = muy sesgada a la derecha
    'max_entre_mediana': (df[gasto].max() / df[gasto].median()).round(1),
})
sesgo

,asimetria,max_entre_mediana
frescos,2.56,13.2
lacteos,4.05,20.3
abarrotes,3.59,19.5
congelados,5.91,39.9
detergentes_papel,3.63,50.0
delicatessen,11.15,49.7


✍️ **¿Qué observas?** ¿Qué forma tienen los histogramas? ¿Qué variable es la más sesgada? ¿Cuántas veces la mediana gasta el cliente más grande? Si calculáramos distancias con estos datos, ¿qué clientes dominarían?

Los histogramas son fuertemente sesgados a la derecha, especialmente delicatessen. Los máximos llegan a unas 50 veces la mediana en algunas categorías; sin escala o transformación dominarían las distancias.

In [9]:
plot_correlation_matrix(df, gasto)


✍️ **¿Qué observas?** ¿Qué pares de variables están más correlacionados? ¿Ves un "bloque" de variables que suben juntas? ¿Qué tipo de negocio compraría esa canasta? ¿Por qué esto anticipa que PCA servirá?

Los gastos de varias categorías están correlacionados y forman canastas relacionadas. Esto hace probable que PCA resuma varias categorías en pocos ejes generales.

---
# 2️⃣ ¿Por qué `log1p` antes de escalar? (20 min)

`StandardScaler` resta la media y divide entre la desviación estándar, pero **no cambia la forma** de la distribución: un cliente que estaba 16 desviaciones arriba sigue igual de lejos. Para datos de dinero, el truco clásico es aplicar el **logaritmo** antes de escalar:

$$x' = \log(1 + x)$$

- El logaritmo **comprime la cola**: pasar de 1,000 a 10,000 pesa lo mismo que pasar de 10,000 a 100,000 (multiplicar por 10).
- Usamos `log1p` (es decir, $\log(1+x)$) en vez de `log` para que un gasto de 0 no dé $-\infty$.
- En lenguaje de negocio: comparamos **proporciones** ("gasta el doble") en vez de diferencias absolutas ("gasta 5,000 más").

Vamos a escalar **de las dos formas** y comparar.

In [10]:
df_log = df.copy()
for col in gasto:
    df_log[col] = np.log1p(df[col])
plot_distributions(df_log, gasto)


In [11]:
# Esta celda va de regalo: léela con calma y ejecútala
# Asimetría antes y después del logaritmo
pd.DataFrame({
    'asimetria_original': df[gasto].skew().round(2),
    'asimetria_log': df_log[gasto].skew().round(2),
})

,asimetria_original,asimetria_log
frescos,2.56,-1.58
lacteos,4.05,-0.22
abarrotes,3.59,-0.67
congelados,5.91,-0.35
detergentes_papel,3.63,-0.24
delicatessen,11.15,-1.09


✍️ **¿Qué observas?** ¿Cómo cambió la forma de los histogramas? Compara la asimetría antes y después. ¿Alguna variable quedó sesgada hacia el otro lado? ¿Qué clientes crees que están ahora en esa cola izquierda?

La asimetría baja tras el log: frescos pasa de 2.56 a -1.58 y delicatessen de 11.15 a -1.09. Los clientes de gasto bajo quedan visibles en una cola izquierda.

In [12]:
from sklearn.preprocessing import StandardScaler
scaler_crudo = StandardScaler(); X_crudo = scaler_crudo.fit_transform(df[gasto])
scaler_log = StandardScaler(); X_log = scaler_log.fit_transform(df_log[gasto])
print('X_crudo:', X_crudo.shape)
print('X_log:', X_log.shape)


X_crudo: (440, 6)
X_log: (440, 6)


In [13]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Qué tan lejos queda el cliente más extremo de cada variable? (en desviaciones estándar)
pd.DataFrame({
    'max_|z|_crudo': np.abs(X_crudo).max(axis=0).round(1),
    'max_|z|_log': np.abs(X_log).max(axis=0).round(1),
    'celdas_|z|>3_crudo': (np.abs(X_crudo) > 3).sum(axis=0),
    'celdas_|z|>3_log': (np.abs(X_log) > 3).sum(axis=0),
}, index=gasto)

,max_|z|_crudo,max_|z|_log,celdas_|z|>3_crudo,celdas_|z|>3_log
frescos,7.9,5.0,7,6
lacteos,9.2,3.8,9,2
abarrotes,8.9,6.4,7,2
congelados,11.9,3.2,6,1
detergentes_papel,8.0,3.2,10,2
delicatessen,16.5,4.1,4,8


### 🧪 Prueba rápida: ¿cambia el agrupamiento?

Ajustamos un K-Means con k = 4 en cada espacio (un valor cualquiera, sólo para comparar; el k definitivo lo elegimos en la Parte 4) y vemos **cuántos clientes caen en cada cluster**.

In [14]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.cluster import KMeans

kmeans_crudo = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_crudo = kmeans_crudo.fit_predict(X_crudo)

print('Tamaños con datos crudos:', np.bincount(labels_crudo))
print('Silhouette (espacio crudo):', evaluar_clustering('crudo', X_crudo, labels_crudo)['silhouette'])

Tamaños con datos crudos: [110 315  10   5]
Silhouette (espacio crudo): 0.348


In [15]:
kmeans_log = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_log = kmeans_log.fit_predict(X_log)
print('Tamaños:', np.bincount(labels_log))
print('Silhouette:', evaluar_clustering('log', X_log, labels_log)['silhouette'])


Tamaños: [ 61 129 132 118]
Silhouette: 0.188


✍️ **¿Qué observas?** ¿Qué tan lejos queda el cliente más extremo en cada versión? Compara los tamaños de los clusters: ¿cuál te serviría para organizar equipos de venta? La silhouette del espacio crudo sale más alta: ¿qué está premiando? ¿Por qué no deberías compararla con la silhouette calculada en el espacio log?

El modelo crudo tiene tamaños muy desbalanceados (110, 315, 10, 5) y silhouette 0.348; el modelo log da tamaños más equilibrados (61, 129, 132, 118) y 0.188. La silhouette cruda está inflada por separación de extremos y no debe compararse directamente.

---
# 3️⃣ PCA: de 6 categorías a 2 ejes con significado (25 min)

Vimos que varias categorías se mueven juntas. PCA busca las direcciones que concentran la mayor varianza; si dos o tres ejes resumen casi todo, podemos **agrupar y graficar** en ese espacio reducido. Primero ajustamos PCA con **todos** los componentes para decidir cuántos quedarnos.

In [16]:
from sklearn.decomposition import PCA
pca_log = PCA().fit(X_log)
print('Varianza:', pca_log.explained_variance_ratio_)
print('Acumulada:', np.cumsum(pca_log.explained_variance_ratio_))
plot_varianza_explicada(pca_log)


Varianza: [0.44077752 0.27194917 0.10698428 0.10103287 0.04876269 0.03049347]
Acumulada: [0.44077752 0.71272669 0.81971097 0.92074384 0.96950653 1.        ]


✍️ **¿Qué observas?** ¿Cuánta varianza explican PC1 y PC2 juntos? ¿Dónde ves el codo? ¿Cuántos componentes necesitarías para el 80%? ¿Con cuántos te quedarías y por qué?

PC1-PC2 explican 71.27%; con tres componentes se alcanza 81.97%. El codo está alrededor de PC2-PC3 y conservaría tres componentes.

In [17]:
plot_cargas(pca_log, gasto, n_componentes=3)


✍️ **¿Qué observas?** Completa la tabla con las variables que más pesan en cada componente y ponle un nombre de negocio a cada eje.

| Componente | Variables con más peso | Nombre de negocio |
|---|---|---|
| PC1 | ______ | ______ |
| PC2 | ______ | ______ |

| PC1 | Frescos, lácteos, abarrotes y delicatessen | Volumen general |
| PC2 | Abarrotes/lácteos frente a frescos/delicatessen | Especialización de canasta |

In [18]:
n_componentes = int(np.argmax(np.cumsum(pca_log.explained_variance_ratio_) >= 0.80) + 1)
pca_final = PCA(n_components=n_componentes)
X_pcs = pca_final.fit_transform(X_log)
df_pcs = pd.DataFrame(X_pcs, columns=[f'PC{i + 1}' for i in range(X_pcs.shape[1])], index=df.index)
print('Componentes elegidos:', n_componentes)
plot_biplot(X_pcs, pca_final, gasto)


Componentes elegidos: 3


✍️ **¿Qué observas?** ¿Hacia dónde apuntan las flechas de cada canasta? ¿Ves grupos claramente separados en la nube o una sola masa? ¿Hay clientes lejos del resto? ¿Qué significa estar arriba a la derecha del biplot?

Las flechas apuntan a las canastas. Arriba a la derecha significa altos gastos en varias categorías; hay una masa continua y varios clientes extremos.

---
# ☕ Descanso (10 min)

---
# 4️⃣ K-Means sobre los componentes principales (25 min)

Ahora sí, agrupamos en el espacio `X_pcs` (los componentes que elegiste). Como en la sesión 1, elegimos `k` con **varias señales**: codo, silhouette y gráfica de silueta... y con **sentido de negocio**.

In [19]:
tabla_k = tabla_metricas_k(X_pcs, range(2, 9))
tabla_k


,k,inercia,silhouette
0,2,1368.4,0.356
1,3,1078.1,0.331
2,4,914.2,0.255
3,5,810.5,0.277
4,6,717.0,0.286
5,7,639.4,0.278
6,8,588.7,0.282


In [20]:
plot_codo(tabla_k['k'], tabla_k['inercia'])
plot_silueta_k(tabla_k)


✍️ **¿Qué observas?** ¿Dónde ves el codo? ¿Qué k prefiere la silhouette? ¿Coinciden? Si los candidatos casi empatan, ¿qué criterio usarías para decidir? Escribe el k que eliges y por qué.

k=2 tiene la mayor silhouette (0.356), frente a 0.331 en k=3 y 0.255–0.286 en k=4–8. Por estabilidad y simplicidad elegiría k=2.

In [21]:
k_elegido = int(tabla_k.loc[tabla_k['silhouette'].idxmax(), 'k'])
kmeans_final = KMeans(n_clusters=k_elegido, n_init=10, random_state=42)
labels_final = kmeans_final.fit_predict(X_pcs)
print('Tamaños:', np.bincount(labels_final))
print('k elegido:', k_elegido)
plot_clusters(df_pcs, 'PC1', 'PC2', labels_final, centroides=kmeans_final.cluster_centers_) 


Tamaños: [188 252]
k elegido: 2


In [22]:
plot_silueta(X_pcs, labels_final)


In [23]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Perdimos algo por agrupar en los componentes principales y no en las 6 variables (log + escaladas)?
kmeans_6d = KMeans(n_clusters=kmeans_final.n_clusters, n_init=10, random_state=42)
labels_6d = kmeans_6d.fit_predict(X_log)

# Tabla cruzada: filas = grupo usando los PCs, columnas = grupo usando las 6 variables
# (los números de cluster son arbitrarios: el "0" de un modelo puede ser el "1" del otro)
tabla_pcs_vs_6d = pd.crosstab(
    pd.Series(labels_final, name='K-Means en PCs'),
    pd.Series(labels_6d, name='K-Means en 6 variables'),
)
# Para cada grupo de los PCs, la celda más grande son los clientes que "se quedan juntos"
print('Clientes que quedan en el mismo grupo:', tabla_pcs_vs_6d.max(axis=1).sum(), 'de', len(labels_final))
tabla_pcs_vs_6d

Clientes que quedan en el mismo grupo: 440 de 440


K-Means en 6 variables,0,1
K-Means en PCs,,
0,0,188
1,252,0


✍️ **¿Qué observas?** ¿Cuántos clientes quedaron en cada cluster? ¿Dónde está cada cluster en el plano PC1-PC2 y qué significa eso en términos de canastas? ¿La silueta es alta o moderada? ¿Por qué? ¿Perdimos información al agrupar con los componentes en vez de las 6 variables?

El modelo tiene 188 clientes en el cluster 0 y 252 en el cluster 1. El 0 compra más abarrotes/lácteos y el 1 más frescos. La silhouette 0.356 es moderada. La asignación coincide con el modelo de seis variables y tres PCs conservan 82% de la varianza.

### 📊 Guardamos los resultados

In [24]:
comparacion = []
comparacion.append(evaluar_clustering('K-Means final (PCs)', X_pcs, labels_final))
comparacion


[{'modelo': 'K-Means final (PCs)',
  'n_clusters': 2,
  'pct_ruido': 0.0,
  'silhouette': 0.356}]

---
# 5️⃣ DBSCAN: ¿quién no encaja en ningún segmento? (20 min)

K-Means **obliga** a cada cliente a pertenecer a un cluster, aunque esté lejísimos de todos. DBSCAN no: lo que no está en una zona densa queda como **ruido**. Lo usamos sobre el **mismo espacio** `X_pcs` para encontrar clientes atípicos.

Receta de la sesión 2:
1. `min_samples` ≈ 2 × número de dimensiones. Calcúlalo con los componentes que elegiste: `min_samples = 2 * X_pcs.shape[1]`.
2. `eps` sale de la **gráfica de k-distancia** con k = `min_samples`: buscamos el codo donde la curva se dispara.

In [25]:
min_samples = 2 * X_pcs.shape[1]
plot_k_distancia(X_pcs, k=min_samples)


✍️ **¿Qué observas?** ¿En qué valor de la distancia empieza a dispararse la curva? ¿Qué `eps` elegiste? ¿Qué porcentaje aproximado de clientes queda por encima de tu línea?

La curva empieza a dispararse alrededor de 0.4–0.6. Elegí eps=0.4; encuentra seis grupos y 62.7% de ruido.

In [26]:
from sklearn.cluster import DBSCAN
eps_elegido = 0.4
dbscan_atipicos = DBSCAN(eps=eps_elegido, min_samples=min_samples)
labels_db = dbscan_atipicos.fit_predict(X_pcs)
fila_db = evaluar_clustering('DBSCAN (PCs)', X_pcs, labels_db)
comparacion.append(fila_db)
fila_db


{'modelo': 'DBSCAN (PCs)',
 'n_clusters': 6,
 'pct_ruido': 62.7,
 'silhouette': 0.315}

In [27]:
plot_clusters(df_pcs, 'PC1', 'PC2', labels_db, title='DBSCAN sobre PC1 y PC2')


In [28]:
# Esta celda va de regalo: léela con calma y ejecútala
# Sensibilidad: ¿qué pasa con otros valores de eps?
pd.DataFrame([
    evaluar_clustering(f'eps={eps}', X_pcs, DBSCAN(eps=eps, min_samples=dbscan_atipicos.min_samples).fit_predict(X_pcs))
    for eps in [0.4, 0.6, 0.8, 1.0, 1.2]
])

,modelo,n_clusters,pct_ruido,silhouette
0,eps=0.4,6,62.7,0.315
1,eps=0.6,2,23.2,0.318
2,eps=0.8,1,13.4,NaN
3,eps=1.0,1,8.4,NaN
4,eps=1.2,1,5.7,NaN


✍️ **¿Qué observas?** ¿Cuántos clusters encontró DBSCAN y qué porcentaje es ruido? ¿Por qué la silhouette sale `NaN`? ¿Qué te dice esto sobre si los clientes forman grupos separados o un continuo? Según la tabla de sensibilidad, ¿qué tan estable es el porcentaje de ruido?

Con eps=0.4 DBSCAN encuentra seis grupos y deja 62.7% como ruido. La silhouette sobre los densos es 0.315; eps mayores hacen que casi desaparezcan los grupos y sugieren un continuo más que clústeres separados.

### 🔎 ¿Quiénes son los atípicos?

Miramos a los clientes marcados como ruido **en unidades originales** (u.m. al año), ordenados por su gasto total.

In [29]:
# Esta celda va de regalo: léela con calma y ejecútala
df['gasto_total'] = df[gasto].sum(axis=1)

atipicos = df[labels_db == -1].sort_values('gasto_total', ascending=False)
atipicos[gasto + ['gasto_total', 'canal']]

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total,canal
85,16117,46197,92780,1026,40827,2944,199891,Retail
47,44466,54259,55571,7782,24171,6465,192714,Retail
181,112151,29627,18148,16745,4948,8550,190169,Horeca
183,36847,43950,20170,36534,239,47943,185683,Horeca
61,35942,38369,59598,3254,26701,2017,165881,Retail
...,...,...,...,...,...,...,...,...
131,2101,589,314,346,70,310,3730,Horeca
355,190,727,2012,245,184,127,3485,Horeca
98,503,112,778,895,56,132,2476,Horeca
97,403,254,610,774,54,63,2158,Horeca


✍️ **¿Qué observas?** Mira a los clientes atípicos: ¿todos son grandes compradores? ¿Ves algún patrón distinto entre ellos? ¿Qué tipos de atípicos distingues? ¿Por qué el logaritmo hizo visibles a algunos de ellos?

Los atípicos incluyen grandes compradores de distintas canastas y algunos gastos pequeños. Hay extremos de frescos/abarrotes, lácteos/papel y delicatessen. El log redujo su dominancia y permitió verlos.

---
# 6️⃣ Perfil de los segmentos en unidades originales (20 min)

Un cluster sólo es útil si se puede **explicar en el idioma del negocio**. Nadie en el área comercial entiende "PC1 = 1.18": hay que regresar a **u.m. al año**. Por eso aplicamos `perfil_clusters` sobre el `df` **original**, sin logaritmo ni escalado.

In [30]:
perfil_final = perfil_clusters(df, labels_final, gasto + ['gasto_total'])
perfil_final


,n,pct,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total
Cluster 0,188,42.7,9355.9,10346.4,14697.1,2222.5,6084.5,2177.4,44883.7
Cluster 1,252,57.3,13973.1,2401.8,2918.7,3705.7,491.9,1038.0,24529.2


In [31]:
# Esta celda va de regalo: léela con calma y ejecútala
# Como los gastos son muy sesgados, la mediana describe mejor al cliente típico de cada cluster
medianas = df[gasto + ['gasto_total']].groupby(labels_final).median().round(0)
medianas.index = [f'Cluster {c}' for c in medianas.index]

# ¿Qué porcentaje de la venta total aporta cada cluster?
medianas['pct_de_la_venta'] = (
    100 * df.groupby(labels_final)['gasto_total'].sum() / df['gasto_total'].sum()
).round(1).values
medianas

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total,pct_de_la_venta
Cluster 0,5406.0,7690.0,11527.0,1062.0,4495.0,1388.0,35222.0,57.7
Cluster 1,10348.0,1871.0,2406.0,2225.0,328.0,750.0,19745.0,42.3


✍️ **¿Qué observas?** Para cada cluster: ¿en qué categorías gasta más y en cuáles casi nada? ¿Cuál es el más grande en número de clientes? ¿Cuál aporta más venta? Ponle un **nombre de negocio** a cada segmento.

| Cluster | Nombre del segmento | ¿Qué lo distingue? |
|---|---|---|
| ______ | ______ | ______ |
| ______ | ______ | ______ |

(Agrega una fila por cada cluster de tu modelo.)

| 0 | Abarrotes/lácteos | 188 clientes; 57.7% de venta; abarrotes 14.7k y lácteos 10.3k |
| 1 | Compradores de frescos | 252 clientes; 42.3% de venta; mayor gasto en frescos |

In [32]:
perfil_clusters(df, labels_db, gasto + ['gasto_total'])


,n,pct,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total
Ruido,276,62.7,11364.7,6696.6,9333.5,3011.8,3544.4,1667.1,35618.0
Cluster 0,27,6.1,11186.4,6949.6,9565.3,1864.4,3244.7,1954.1,34764.5
Cluster 1,104,23.6,14134.1,2075.7,2691.1,3473.7,447.9,936.9,23759.4
Cluster 2,9,2.0,11907.1,11860.4,17818.2,1915.8,7913.2,1711.6,53126.3
Cluster 3,10,2.3,6372.8,3999.2,4328.5,1844.9,823.7,1503.8,18872.9
Cluster 4,9,2.0,17337.0,5012.6,5579.2,7319.7,865.1,2565.4,38679.0
Cluster 5,5,1.1,8915.6,21348.8,26102.8,1442.6,13635.6,1419.4,72864.8


In [33]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿En qué segmento de K-Means había quedado cada atípico?
pd.crosstab(
    pd.Series(labels_db, name='DBSCAN').map(lambda c: 'Ruido' if c == -1 else 'Denso'),
    pd.Series(labels_final, name='K-Means')
)

K-Means,0,1
DBSCAN,,
Denso,42,122
Ruido,146,130


✍️ **¿Qué observas?** ¿El promedio del ruido describe bien a los atípicos? ¿Por qué no? ¿En qué clusters de K-Means habían quedado? ¿Qué pasaría si no los separáramos?

El promedio del ruido mezcla 276 clientes y oculta perfiles. 146 estaban en el cluster 1 de K-Means y 130 en el cluster 0. Separarlos evita que extremos distorsionen los perfiles.

---
# 7️⃣ Validación contra el canal de venta (15 min)

Ahora sí sacamos la variable que guardamos: `canal` (Horeca o Retail). **No la usamos para agrupar**, así que es una prueba honesta: si nuestros segmentos tienen sentido, deberían relacionarse con el tipo de negocio del cliente.

⚠️ En la mayoría de los proyectos reales **no tienes** una etiqueta así. Aquí tenemos la suerte de poder revisar; aun así, no buscamos que los clusters **copien** el canal: buscamos que lo **expliquen** y, si se puede, que digan algo más.

In [34]:
pd.crosstab(df['canal'], labels_final, rownames=['canal'], colnames=['cluster'], margins=True)


cluster,0,1,All
canal,,,
Horeca,54,244,298
Retail,134,8,142
All,188,252,440


In [35]:
kmeans_k2 = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_k2 = kmeans_k2.fit_predict(X_pcs)
comparacion.append(evaluar_clustering('K-Means k=2 (PCs)', X_pcs, labels_k2))
pd.crosstab(df['canal'], labels_k2, rownames=['canal'], colnames=['cluster'], margins=True)


cluster,0,1,All
canal,,,
Horeca,54,244,298
Retail,134,8,142
All,188,252,440


In [36]:
# Esta celda va de regalo: léela con calma y ejecútala
# Mismas tablas, pero en % de cada cluster (cada columna suma 100)
for nombre, labels in [('K-Means final', labels_final), ('K-Means k=2', labels_k2)]:
    print(f'--- {nombre}: % de cada cluster por canal ---')
    print((100 * pd.crosstab(df['canal'], labels, rownames=['canal'], colnames=['cluster'], normalize='columns')).round(1))

# ¿Cuántos clientes quedan en un cluster cuyo canal mayoritario NO es el suyo?
# (en cada columna, todo lo que no está en la celda más grande)
for nombre, labels in [('K-Means final', labels_final), ('K-Means k=2', labels_k2)]:
    tabla = pd.crosstab(df['canal'], labels)
    print(f'{nombre}: {len(labels) - tabla.max(axis=0).sum()} de {len(labels)} clientes fuera del canal mayoritario de su cluster')

--- K-Means final: % de cada cluster por canal ---
cluster     0     1
canal              
Horeca   28.7  96.8
Retail   71.3   3.2
--- K-Means k=2: % de cada cluster por canal ---
cluster     0     1
canal              
Horeca   28.7  96.8
Retail   71.3   3.2
K-Means final: 62 de 440 clientes fuera del canal mayoritario de su cluster
K-Means k=2: 62 de 440 clientes fuera del canal mayoritario de su cluster


In [37]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Y la región? Comparamos la mezcla de regiones de cada cluster con la del total
mezcla_region = 100 * pd.crosstab(df['region'], labels_final, rownames=['region'], colnames=['cluster'], normalize='columns')
mezcla_region['Total'] = 100 * df['region'].value_counts(normalize=True)
mezcla_region.round(1)

cluster,0,1,Total
region,,,
Lisboa,15.4,19.0,17.5
Oporto,10.1,11.1,10.7
Otra,74.5,69.8,71.8


In [38]:
plot_biplot(X_pcs, pca_final, gasto, labels=df['canal'], title='Biplot coloreado por canal')


✍️ **¿Qué observas?** ¿Algún cluster es casi puro de un canal? ¿Qué % de cada uno de tus segmentos es Retail? ¿Cuál de los modelos se parece más al canal (cuántos clientes quedan fuera del canal mayoritario de su cluster)? Si tu modelo final se parece menos al canal, ¿significa que está mal? ¿La mezcla de regiones cambia de un cluster a otro? ¿Qué te dice eso sobre organizar a los vendedores por zona? En el biplot, ¿hacia dónde se cargan los Retail y los Horeca?

El cluster 0 es 71.3% Retail y el cluster 1 96.8% Horeca; 62 clientes quedan fuera del canal mayoritario. El modelo valida una separación por canal, pero no usa la etiqueta. La mezcla regional cambia poco.

---
# 8️⃣ Recomendaciones de negocio y cierre (15 min)

El último paso del flujo es **actuar**. Para cada segmento decide: qué **equipo de venta** lo atiende, qué **surtido** se le ofrece y qué **logística** necesita. Y no olvides a los atípicos.

### 💼 Recomendaciones por segmento

✍️ Completa la tabla con **tus** segmentos (los nombres que elegiste en la Parte 6) y tus números.

| Segmento | Clientes | % de la venta | Equipo de venta | Surtido | Logística |
|---|---|---|---|---|---|
| ______ | ______ | ______ | ______ | ______ | ______ |
| ______ | ______ | ______ | ______ | ______ | ______ |
| Atípicos (DBSCAN) | ______ | — | ______ | ______ | ______ |

(Agrega una fila por cada segmento de tu modelo.)

✍️ ¿Qué harías con los clientes atípicos? ¿Les darías el mismo trato a todos?

| Abarrotes/lácteos | 188 | 57.7% | Consultivo mayorista | Abarrotes, lácteos y papel | Rutas frecuentes |
| Compradores de frescos | 252 | 42.3% | Comercial Horeca/frescos | Frescos y congelados | Cadena fría |
| Atípicos | 276 | Por revisar | Cuenta clave individual | Personalizado | Entregas especiales |
Los atípicos no deben recibir el mismo trato: conviene contar con atención individual para grandes cuentas y revisar casos pequeños o anomalías.

### 📊 Tabla comparativa final

In [39]:
tabla_comparacion = pd.DataFrame(comparacion)
print(tabla_comparacion)


                modelo  n_clusters  pct_ruido  silhouette
0  K-Means final (PCs)           2        0.0       0.356
1         DBSCAN (PCs)           6       62.7       0.315
2    K-Means k=2 (PCs)           2        0.0       0.356


✍️ **¿Qué observas?** ¿Cuál modelo gana en cada métrica? ¿Por qué sí podemos comparar estas silhouettes entre sí (a diferencia de la del espacio crudo)? ¿Qué aporta DBSCAN si no tiene silhouette?

El log y PCA con tres componentes conservan 82% de la varianza; K-Means k=2 da los segmentos más simples con silhouette 0.356. DBSCAN identifica atípicos útiles para cuentas especiales. Aprendí a controlar extremos, comprimir variables y usar canales sólo como validación.

## ❓ Respuesta a la pregunta

> **¿Cómo debe organizar el distribuidor a sus clientes mayoristas?**

✍️ Escribe tu respuesta en 4-6 líneas: cuántos segmentos, cómo se llaman, cuántos clientes y qué % de la venta trae cada uno, qué equipo/surtido/logística le toca, qué hacer con los atípicos y si la región o el canal bastarían para organizarlos.

El log y PCA con tres componentes conservan 82% de la varianza; K-Means k=2 da los segmentos más simples con silhouette 0.356. DBSCAN identifica atípicos útiles para cuentas especiales. Aprendí a controlar extremos, comprimir variables y usar canales sólo como validación.

## 📝 Lo que aprendimos

✍️ Escribe las 3 a 5 lecciones más importantes de este caso (y del curso).

El log y PCA con tres componentes conservan 82% de la varianza; K-Means k=2 da los segmentos más simples con silhouette 0.356. DBSCAN identifica atípicos útiles para cuentas especiales. Aprendí a controlar extremos, comprimir variables y usar canales sólo como validación.

## 🚀 Retos opcionales

1. Repite el análisis con **un componente más** de los que elegiste. ¿Cambian los segmentos?
2. Prueba **K-Means con otro k**. ¿El segmento nuevo es accionable o sólo parte uno existente?
3. Usa **HDBSCAN** (`sklearn.cluster.HDBSCAN`) sobre `X_pcs`. ¿Encuentra los mismos atípicos?
4. Agrupa con **proporciones** en vez de montos: divide cada gasto entre el `gasto_total` del cliente. ¿Aparecen segmentos por *tipo* de canasta sin importar el tamaño?